# Notebook 1: Extract and Score Raw Texts

This notebook processes raw `.txt` files sorted by CEFR level, and adds basic readability metrics (Flesch-Kincaid Grade and SMOG index).

It exports a structured CSV that will be used for further analysis and modeling in later notebooks.

## 1. Setup

Install required packages and import libraries.


In [ ]:
import os
import pandas as pd
import textstat

## 2. Load Raw Texts by CEFR Level

Each CEFR level (A1–C2) is mapped to an estimated age group. The script reads and stores each text file into a DataFrame.


In [ ]:
cefr_to_age = {
    "A1": "6-8",
    "A2": "8-10",
    "B1": "10-12",
    "B2": "12-14",
    "C1": "14-16",
    "C2": "16-18"
}

rows = []
data_folder = "../data/raw_data"

for cefr_folder in cefr_to_age.keys():
    folder_path = os.path.join(data_folder, cefr_folder)
    if not os.path.isdir(folder_path):
        continue
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):   
            filepath = os.path.join(folder_path, filename)
            with open(filepath, encoding="utf-8") as file:
                text = file.read().strip() 
            cefr_level = filename.split("_")[0]
            rows.append({
                "cefr_folder": cefr_folder,
                "title": filename.replace(".txt", ""),
                "cefr_age_group": cefr_to_age[cefr_folder],
                "text": text,
            })

df = pd.DataFrame(rows)


## 3. Calculate Readability Scores

For each text, calculate:

- Flesch-Kincaid Grade Level
- SMOG Index

These will later be used to classify texts into reading bands.


In [ ]:
def calculate_readability_scores(text):
    """
    Calculate Flesch-Kincaid and SMOG readability scores for a given text.
    """
    fk_score = textstat.flesch_kincaid_grade(text)
    smog_score = textstat.smog_index(text)
    return fk_score, smog_score

# Apply the function to the 'text' column and create new columns for the scores
df[['flesch_kincaid_score', 'smog_score']] = df['text'].apply(calculate_readability_scores).apply(pd.Series)

In [ ]:

df.to_csv("../preprocessing_data/raw_texts_all_levels_with_smog_and_flesch_kincaid_scores.csv", index=False)

---

📤 **Next step:** See `02_clean_and_organise_dataset.ipynb` to clean, band, and prepare the data for analysis or ML modeling.
